In [ ]:
from convergence_test_helpers import run_convergence_study

In [ ]:
n_refinements = 5
test_cases = ["moeller.xml", "buffa.xml", "vortexflux.xml"]

EXECUTABLE = "../build/bin/stokesConvergenceStudy"

def exectuable_command_func(href, pref):
    return [
        EXECUTABLE,
        # "--no-plot",
        f"-r {href}",
        f"-p {pref}"
    ]

for xml_file in test_cases:
    run_convergence_study(
        executable_command_func=exectuable_command_func,
        xml_file=xml_file,
        n_refinements=n_refinements,
        p_refinements=[0,1,2]
    )

In [ ]:
n_refinements = 5
test_cases = ["moeller.xml", "buffa.xml", "vortexflux.xml"]

EXECUTABLE = "../build/bin/stokesConvergenceStudy"

def exectuable_command_func(href, pref):
    return [
        EXECUTABLE,
        # "--no-plot",
        f"-r {href}",
        f"-p {pref}"
    ]

for xml_file in test_cases:
    run_convergence_study(
        executable_command_func=exectuable_command_func,
        xml_file=xml_file,
        n_refinements=n_refinements,
        p_refinements=[0,1,2]
    )

## `gsIncompressibleFlow` tests

In [ ]:
from convergence_test_helpers import run_convergence_study

n_refinements = 5
test_cases = ["moeller.xml", "buffa.xml", "vortexflux.xml"]
npz_error_files = ["moeller_errors", "buffa_errors", "vortexflux_errors"]

EXECUTABLE = "../ninja-build/bin/stokes_INS_example"

def exectuable_command_func(href, pref):
    return [
        EXECUTABLE,
        "--no-plot",
        f"-r {href}",
        f"-e {pref}"
    ]

for xml_file,npz_error_file in zip(test_cases, npz_error_files):
    run_convergence_study(
        executable_command_func=exectuable_command_func,
        xml_file=xml_file,
        n_refinements=n_refinements,
        p_refinements=[0,1,2, 3,4,5,6,7,8,9],
        npz_outfile=npz_error_file
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plot_slopes = False
save_plot = False

for error_file in npz_error_files:
    errors = np.load(error_file + ".npz")
    velocity_errors = errors["velocity"]
    pressure_errors = errors["pressure"]
    element_sizes = np.power(2.0, -np.arange(velocity_errors.shape[1]))

    # Prepare plot
    fig,axes = plt.subplots(ncols=2, figsize=(8,4), sharey=True)
    ylabels = [
        r"$|| \boldsymbol{v} - \boldsymbol{v}_{\text{ana}} ||_{L_2}$",
        r"$|| p - p_{\text{ana}} ||_{L_2}$"
    ]

    for i in range(velocity_errors.shape[0]):
        velocity_order = i + 2
        vel_errors = velocity_errors[i,:]
        p_errors = pressure_errors[i,:]
        for axis,graph_value,ylabel in zip(
            axes,
            [vel_errors, p_errors],
            ylabels
        ):
            axis.loglog(element_sizes, graph_value, "^-", label=f"Vel. order = {velocity_order}")
            axis.set_xlabel("Element size")
            axis.set_ylabel(ylabel)
            # axis.set_aspect("equal")
            if plot_slopes:
                convergence_rates = np.diff(np.log10(graph_value)) / np.diff(np.log10(element_sizes))
                median_convergence = np.round(np.median(convergence_rates))
                scaling_factor = 0.2*graph_value[-1]/np.power(element_sizes[-1], median_convergence)
                slope_values = scaling_factor*np.power(element_sizes, median_convergence)
                axis.plot(element_sizes, slope_values, "k--", alpha=0.5)
                triangle = plt.Polygon(np.column_stack((
                    [element_sizes[-1], element_sizes[-2], element_sizes[-2], element_sizes[-1]],
                    [slope_values[-1], slope_values[-1], slope_values[-2], slope_values[-1]]
                )), fill=False, zorder=99)
                axis.add_patch(triangle)
                middle_scaling = np.log10(5)
                xtext = np.average(element_sizes[-3:-1], weights=[1.0-middle_scaling, middle_scaling])
                ytext = np.average(slope_values[-2:], weights=[1.0-middle_scaling, middle_scaling])
                axis.annotate(rf"$\mathcal{{O}}({int(median_convergence)})$", xy=(xtext, ytext), ha="center", va="center")
            
    plt.legend()
    fig.suptitle(error_file)
    plt.tight_layout()
    if save_plot:
        plt.savefig(f"{error_file}.svg")
    plt.show()